In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

if not os.path.exists("../output"):
    os.makedirs("../output")

In [ ]:
df = pd.read_csv("../data/cleaned_orders.csv", parse_dates=['下单时间'])
print(f"✅ 成功加载数据：{df.shape[0]} 行")

In [ ]:
top10_products = df.groupby('商品编号')['付款金额'].sum().sort_values(ascending=False).head(10)
platform_sales = df.groupby('平台类型')['付款金额'].sum()
daily_sales = df.groupby(df['下单时间'].dt.date)['付款金额'].sum()
top10_users = df.groupby('用户名')['付款金额'].sum().sort_values(ascending=False).head(10)

In [ ]:
# 图1: 商品销售额 TOP10 (单独保存)
fig1, ax1 = plt.subplots(figsize=(12, 6))
bars1 = ax1.bar(top10_products.index.astype(str), top10_products.values, color='#FF6B6B')
ax1.bar_label(bars1, fmt='%.0f', fontsize=9, padding=3)
ax1.set_title('商品销售额 TOP10', fontsize=16, fontweight='bold')
ax1.set_ylabel('销售额 (元)', fontsize=12)
ax1.set_xlabel('商品编号', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../output/05_01_商品销售额TOP10.png', dpi=300, bbox_inches='tight')
print('✅ 图1已保存: 05_01_商品销售额TOP10.png')
plt.show()

In [ ]:
# 图2: 各平台销售额占比 (单独保存)
platform_pct = platform_sales / platform_sales.sum() * 100
major_platforms = platform_sales[platform_pct >= 5].copy()
other_sum = platform_sales[platform_pct < 5].sum()
if other_sum > 0:
    major_platforms['其他'] = other_sum

def make_autopct(values):
    def my_autopct(pct):
        return f'{pct:.1f}%%' if pct >= 1 else ''
    return my_autopct

fig2, ax2 = plt.subplots(figsize=(8, 8))
ax2.pie(major_platforms.values, labels=major_platforms.index,
        autopct=make_autopct(major_platforms.values), startangle=90,
        colors=['#FF9999','#66B2FF','#99FF99','#FFCC99','#C0C0C0'],
        textprops={'fontsize': 13})
ax2.set_title('各平台销售额占比', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/05_02_平台销售额占比.png', dpi=300, bbox_inches='tight')
print('✅ 图2已保存: 05_02_平台销售额占比.png')
plt.show()

In [ ]:
# 图3: 每日销售趋势 (单独保存)
fig3, ax3 = plt.subplots(figsize=(14, 6))
ax3.plot(daily_sales.index, daily_sales.values, marker='o', color='#4ECDC4', linewidth=1, markersize=2, alpha=0.4, label='原始数据')
ma7 = daily_sales.rolling(window=7).mean()
ax3.plot(ma7.index, ma7.values, color='#E74C3C', linewidth=2.5, label='7日移动平均线')
ax3.legend(loc='upper right', fontsize=11)
ax3.set_title('每日销售趋势 (含7日平滑)', fontsize=16, fontweight='bold')
ax3.set_ylabel('销售额 (元)', fontsize=12)
ax3.set_xlabel('日期', fontsize=12)
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../output/05_03_每日销售趋势.png', dpi=300, bbox_inches='tight')
print('✅ 图3已保存: 05_03_每日销售趋势.png')
plt.show()

In [ ]:
# 图4: 用户消费金额 TOP10 (单独保存)
fig4, ax4 = plt.subplots(figsize=(11, 6))
bars4 = ax4.barh(top10_users.index, top10_users.values, color='#FFE66D')
ax4.bar_label(bars4, fmt='%.0f', fontsize=9, padding=3)
ax4.set_title('用户消费金额 TOP10', fontsize=16, fontweight='bold')
ax4.set_xlabel('消费金额 (元)', fontsize=12)
ax4.set_ylabel('用户名', fontsize=12)
plt.tight_layout()
plt.savefig('../output/05_04_用户消费TOP10.png', dpi=300, bbox_inches='tight')
print('✅ 图4已保存: 05_04_用户消费TOP10.png')
plt.show()

In [ ]:
# 组合图：四合一 (可选，用于总览)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('电商核心业务分析', fontsize=18, fontweight='bold', y=1.02)

bars1 = axes[0, 0].bar(top10_products.index.astype(str), top10_products.values, color='#FF6B6B')
axes[0, 0].bar_label(bars1, fmt='%.0f', fontsize=8, padding=3)
axes[0, 0].set_title('商品销售额 TOP10', fontsize=15, fontweight='bold')
axes[0, 0].set_ylabel('销售额 (元)')
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].pie(major_platforms.values, labels=major_platforms.index,
               autopct=make_autopct(major_platforms.values), startangle=90,
               colors=['#FF9999','#66B2FF','#99FF99','#FFCC99','#C0C0C0'])
axes[0, 1].set_title('各平台销售额占比', fontsize=15, fontweight='bold')

axes[1, 0].plot(daily_sales.index, daily_sales.values, marker='o', color='#4ECDC4', linewidth=1, markersize=2, alpha=0.4, label='原始数据')
axes[1, 0].plot(ma7.index, ma7.values, color='#E74C3C', linewidth=2.5, label='7日移动平均线')
axes[1, 0].legend(loc='upper right')
axes[1, 0].set_title('每日销售趋势 (含7日平滑)', fontsize=15, fontweight='bold')
axes[1, 0].set_ylabel('销售额 (元)')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

bars4 = axes[1, 1].barh(top10_users.index, top10_users.values, color='#FFE66D')
axes[1, 1].bar_label(bars4, fmt='%.0f', fontsize=8, padding=3)
axes[1, 1].set_title('用户消费金额 TOP10', fontsize=15, fontweight='bold')
axes[1, 1].set_xlabel('消费金额 (元)')

plt.tight_layout()
plt.savefig('../output/05_核心业务分析图_四合一.png', dpi=300, bbox_inches='tight')
print('✅ 四合一图已保存: 05_核心业务分析图_四合一.png')
plt.show()